In [ ]:
# Load data and apply conditions
import gzip
import orjson
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

REV_FILE = '/content/drive/MyDrive/Thesis Code/Beauty_and_Personal_Care.jsonl.gz'
META_FILE = '/content/drive/MyDrive/Thesis Code/meta_Beauty_and_Personal_Care.jsonl.gz'

print("loading metadata...")
meta_records = []
with gzip.open(META_FILE, 'rb') as f:
    for line in f:
        data = orjson.loads(line)
        asin = data.get('parent_asin') or data.get('asin')
        price = data.get('price')

        if asin and price:
            try:
                # Fast string cleaning
                clean_p = float(str(price).replace('$', '').replace(',', '').strip())
                meta_records.append({'product_id': asin, 'price': clean_p})
            except ValueError:
                continue

df_meta = pd.DataFrame(meta_records).drop_duplicates(subset=['product_id'])
valid_asins = set(df_meta['product_id'])

print("Loading reviews...")
rev_records = []

#START_TS_2013 = 1356998400000 END_TS_2013 = 1388534399999
#START_TS_2014 = 1388534400000 END_TS_2014 = 1420070399999
# START_TS_2015 = 1420070400000 END_TS_2015   = 1451606399999
#START_TS_2016 = 1451606400000 END_TS_2016   = 1483228799999
#START_TS_2017 = 1483228800000 END_TS_2017   = 1514764799999
#START_TS_2018 = 1514764800000 END_TS_2018   = 1546300799999
#START_TS_2019 = 1546300800000 END_TS_2019   = 1577836799999
#START_TS_2020 = 1577836800000 END_TS_2020   = 1609459199999
#START_TS_2021 = 1609459200000 END_TS_2021   = 1640995199999
#START_TS_2022 = 1640995200000 END_TS_2022   = 1672531199999
#START_TS_2023 = 1672531200000 END_TS_2023   = 1704067199999
#START_TS_2024 = 1704067200000 END_TS_2024   = 1735689599999

START_TS = 1356998400000
END_TS = 1577836799999
TOTAL_MONTHS = 84

with gzip.open(REV_FILE, 'rb') as f:
    for line in f:
        data = orjson.loads(line)
        timestamp = data.get('timestamp') or data.get('unixReviewTime')
        if not timestamp or not (START_TS <= timestamp <= END_TS):
            continue
        asin = data.get('parent_asin') or data.get('asin')
        if asin not in valid_asins:
            continue
        text = data.get('text', '')
        if not text:
            continue
        rating = data.get('rating')
        helpful_vote = data.get('helpful_vote')
        rev_records.append({
            'product_id': asin,
            'rating': float(rating) if rating is not None else np.nan,
            'timestamp': timestamp,
            'review_text': str(text),
            'helpful_vote': float(helpful_vote) if helpful_vote is not None else np.nan
        })

df_rev = pd.DataFrame(rev_records)
print(f"{len(df_rev):,} reviews found.")

print("Merging metadata and formatting dates...")
# Merge the price directly into the review dataframe now
df_rev = df_rev.merge(df_meta, on='product_id', how='left')

df_rev['datetime'] = pd.to_datetime(df_rev['timestamp'], unit='ms')
df_rev['year_month'] = df_rev['datetime'].dt.to_period('M')

print("Applying filtering conditions for segmentation...")
# Calculate total reviews and active months per product
product_stats = df_rev.groupby('product_id').agg(
    total_reviews=('review_text', 'count'),
    active_months=('year_month', 'nunique')
).reset_index()

MIN_REVIEWS = 200
MAX_REVIEWS = 10000
MIN_ACTIVE_MONTHS = int(TOTAL_MONTHS * 0.8)

valid_products = product_stats[
    (product_stats['total_reviews'] >= MIN_REVIEWS) &
    (product_stats['total_reviews'] <= MAX_REVIEWS) &
    (product_stats['active_months'] >= MIN_ACTIVE_MONTHS)
]

# Keep only the reviews that fit filter
df_golden = df_rev[df_rev['product_id'].isin(valid_products['product_id'])].copy()

print(f"Total reviews kept for LLMs: {len(df_golden):,}")
print(f"Total noisy reviews dropped: {len(df_rev) - len(df_golden):,}")

print("Exporting to Parquet....")
OUTPUT_PATH = '/content/drive/MyDrive/Thesis Code/reviews_2013_2019.parquet'
df_golden.to_parquet(OUTPUT_PATH, index=False)
print(f"Successfully saved to {OUTPUT_PATH}")

In [ ]:
#want to have a more limited dataset since cannot run LLM on a million reviews, takes long
print("Stratified Product Sampling...")
MIN_REVIEWS = 200
MAX_REVIEWS = 10000
MIN_ACTIVE_MONTHS = int(0.8*TOTAL_MONTHS)
valid_products = product_stats[
    (product_stats['total_reviews'] >= MIN_REVIEWS) &
    (product_stats['total_reviews'] <= MAX_REVIEWS) &
    (product_stats['active_months'] >= MIN_ACTIVE_MONTHS)
]

# 1. Devide data into 10 quantiles so we can grab a sample of the products representative for the full dataset
valid_products['volume_stratum'] = pd.qcut(valid_products['total_reviews'], q=10, labels=False)

# 2. Aiming for 400 products...
PRODUCTS_PER_STRATUM = 40
sampled_products = valid_products.groupby('volume_stratum').sample(
    n=PRODUCTS_PER_STRATUM,
    random_state=123
)

# 3. Filter the extracted df to ONLY include the sampled products
df_sampled = df_golden[df_golden['product_id'].isin(sampled_products['product_id'])].copy()

print(f"Products Sampled: {sampled_products['product_id'].nunique()}")
print(f"Total Reviews to run: {len(df_sampled):,}")
print(f"Average reviews per product: {len(df_sampled) / len(sampled_products):.0f}")

print("exporting to sampled dataset...")
OUTPUT_PATH = '/content/drive/MyDrive/Thesis Code/cohort_reviews_sampled_2013_2019.parquet'
#df_sampled.to_parquet(OUTPUT_PATH, index=False)
print(f"Successfully saved to {OUTPUT_PATH}")

In [ ]:
#EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
DATA_PATH = '/content/drive/MyDrive/Thesis Code/cohort_reviews_sampled_2013_2019.parquet'
df = pd.read_parquet(DATA_PATH)

print(f"Total Reviews: {len(df):,}")
print(f"Unique Products: {df['product_id'].nunique():,}")
print(f"Timeline: {df['datetime'].min().date()} to {df['datetime'].max().date()}")

print("\nMissing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

monthly_volume = df.groupby('year_month').size()
monthly_volume.index = monthly_volume.index.astype(str)

plt.figure(figsize=(14, 6))
sns.lineplot(x=monthly_volume.index, y=monthly_volume.values, marker='o', color='teal', linewidth=2.5)
plt.title('Total Review Volume Over Time', fontsize=14)
plt.xlabel('Year-Month', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Seasonality check
df['month_of_year'] = df['datetime'].dt.month
seasonality = df.groupby('month_of_year').size()

plt.figure(figsize=(10, 5))
sns.barplot(x=seasonality.index, y=seasonality.values, palette='coolwarm')
plt.title('Reviews by Month of Year', fontsize=14)
plt.xlabel('Month ', fontsize=12)
plt.ylabel('Total Reviews', fontsize=12)
plt.show()


reviews_per_product = df.groupby('product_id').size()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(reviews_per_product, bins=30, kde=True, ax=axes[0], color='purple')
axes[0].set_title('Reviews per Product')
axes[0].set_xlabel('Total Reviews')
axes[0].set_ylabel('Number of Products')

# Review text and rating checks
df['word_count'] = df['review_text'].apply(lambda x: len(str(x).split()))

print(f"Average Word Count: {df['word_count'].mean():.1f} words")
print(f"Median Word Count:  {df['word_count'].median():.1f} words")
print(f"Max Word Count:     {df['word_count'].max():,} words")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word Count Distribution
sns.histplot(df[df['word_count'] <= 500]['word_count'], bins=50, ax=axes[0], color='darkorange')
axes[0].set_title('Review Word Count Distribution')
axes[0].set_xlabel('Word Count')

# Rating Distribution
sns.countplot(data=df, x='rating', palette='viridis', ax=axes[1])
axes[1].set_title('Distribution of Ratings')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

# Correlations of features
print("\n=== 5. BASE CORRELATION CHECK ===")
corr_cols = ['rating', 'helpful_vote', 'price', 'word_count']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title('Correlation Matrix of Base Features')
plt.tight_layout()
plt.show()